# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [1]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Inicialización y constantes and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [3]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [4]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.get_contents())
frog.links

Título de la Web:
Frogames Formación: cursos de programación en español
Contenido de la Web:
Continuar
Cursos
Ver todo el catálogo
Convalidar curso de Udemy
Rutas
Suscripciones
Recursos
Chuletas
Blog
Podcast
Escuchar en Spotify
Juegos
FrogCards
NUEVO
Minijuegos
Frogames
Instructores
Certificaciones
Comunidad
Mi Portal
Panel de Admin
Mis Cursos
Mis Certificados
Mi Perfil y Logros
Mi Comunidad
Panel de Afiliado
Mi Cuenta
Soporte
Cerrar sesión
Español
English
Acceder
Ir al contenido principal
Frogames
Cursos online de programación, videojuegos, matemáticas e inteligencia artificial
18 rutas de aprendizaje · de cero a experto
245
cursos
5954
horas
708.820
estudiantes
4,99★
4802 reseñas
18
rutas
Rutas de aprendizaje
Cada ruta te lleva de cero a experto con un orden pensado. Elige la tuya y empieza hoy, o mira
por dónde empieza cada una
.
Desarrollo de Videojuegos
Unity, Unreal y Godot, del primer prototipo al juego publicado.
31 cursos · 792,5 h
Android
Kotlin, Jetpack Compose y publicación

['https://cursos.frogamesformacion.com/',
 'https://cursos.frogamesformacion.com/enrollments',
 'https://cursos.frogamesformacion.com/collections',
 'https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy',
 'https://cursos.frogamesformacion.com/pages/rutas',
 'https://cursos.frogamesformacion.com/pages/rutas-de-aprendizaje',
 'https://recursos.frogamesformacion.com/',
 'https://cursos.frogamesformacion.com/pages/blog',
 'https://podcast.frogamesformacion.com/',
 'https://open.spotify.com/show/0TJmupUvuuFsUWGFImZ7Bi',
 'https://juegos.frogamesformacion.com/',
 'https://cursos.frogamesformacion.com/pages/proyectos-evaluacion',
 'https://cursos.frogamesformacion.com/pages/instructores',
 'https://cursos.frogamesformacion.com/pages/certificaciones',
 'https://cursos.frogamesformacion.com/communities/Q29tbXVuaXR5LTIzMjY3',
 'https://cursos.frogamesformacion.com/manage',
 'https://cursos.frogamesformacion.com/enrollments',
 'https://cursos.frogamesformacion.com/pages/

## Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes

### Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about".
Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [5]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [6]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [6]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [7]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://cursos.frogamesformacion.com - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):
https://cursos.frogamesformacion.com/
https://cursos.frogamesformacion.com/enrollments
https://cursos.frogamesformacion.com/collections
https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy
https://cursos.frogamesformacion.com/pages/rutas
https://cursos.frogamesformacion.com/pages/rutas-de-aprendizaje
https://recursos.frogamesformacion.com/
https://cursos.frogamesformacion.com/pages/blog
https://podcast.frogamesformacion.com/
https://open.spotify.com/show/0TJmupUvuuFsUWGFImZ7Bi
https://juegos.frogamesformacion.com/
https://cursos.frogamesformacion.com/pages/proyectos-evaluacion
https://cursos.frogamesformac

In [8]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [9]:
anthropic = Website("https://anthropic.com")
anthropic.links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/research/team/alignment',
 'https://www.anthropic.com/research/team/economics',
 'https://www.anthropic.com/engineering',
 'https://www.anthropic.com/research/team/frontier-red-team',
 'https://www.anthropic.com/research/team/interpretability',
 'https://www.anthropic.com/science',
 'https://www.anthropic.com/research/team/societal-impacts',
 'https://www.anthropic.com/policy',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/claude-corps',
 'https://www.anthropic.com/policy-on-the-ai-exponential',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'https://www.anthropic.com/beneficial-deployments',
 'http://trust.anthropic.com/',
 'https://academy.claude.com',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://platform.claude.com/docs',
 'https://www.a

In [10]:
get_links("https://anthropic.com")

{'links': [{'type': 'Pagina de la empresa',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'Carreras/Empleos disponibles',
   'url': 'https://www.anthropic.com/careers'},
  {'type': 'Pagina Sobre nosotros',
   'url': 'https://www.anthropic.com/company/leadership'},
  {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'},
  {'type': 'Pagina de Noticias', 'url': 'https://www.anthropic.com/news'},
  {'type': 'Pagina de Investigación',
   'url': 'https://www.anthropic.com/research'},
  {'type': 'Pagina de Cursos', 'url': 'https://academy.claude.com'}]}

In [11]:
get_links("https://cursos.frogamesformacion.com")

{'links': [{'type': 'Página de Inicio',
   'url': 'https://cursos.frogamesformacion.com/'},
  {'type': 'Página de Rutas de Aprendizaje',
   'url': 'https://cursos.frogamesformacion.com/pages/rutas-de-aprendizaje'},
  {'type': 'Página de Instructores',
   'url': 'https://cursos.frogamesformacion.com/pages/instructores'},
  {'type': 'Página de Certificaciones',
   'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'},
  {'type': 'Página de Proyectos de Evaluación',
   'url': 'https://cursos.frogamesformacion.com/pages/proyectos-evaluacion'},
  {'type': 'Página de Clientes',
   'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'},
  {'type': 'Página sobre Frogames para Empresas',
   'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'},
  {'type': 'Página de Cursos',
   'url': 'https://cursos.frogamesformacion.com/collections'},
  {'type': 'Página de Premios',
   'url': 'https://cursos.frogamesformacion.com/pages/premios'},
  {'type':

## Segundo paso: ¡crea el folleto!

Reúne todos los detalles en otro mensaje para GPT4-o

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [13]:
print(get_all_details("https://anthropic.com"))

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Liderazgo', 'url': 'https://www.anthropic.com/company/leadership'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}]}
Landing page:
Título de la Web:
Home \ Anthropic
Contenido de la Web:
Skip to main content
Skip to footer
Research
Overview
Alignment
Economics
Engineering
Frontier Red Team
Interpretability
Science
Societal Impacts
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Beneficial Deployments
Trust center
Security and compliance
Learn
Learn
Claude Academy
Tutorials
Use cases
Developer docs
Company
About
Leadership
Care

In [24]:
# system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
# y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
# Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [16]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina Principal', 'url': 'https://www.anthropic.com/'}, {'type': 'Pagina Acerca de', 'url': 'https://www.anthropic.com/company'}, {'type': 'Carreras/Empleos', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina de Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Pagina de Investigación', 'url': 'https://www.anthropic.com/research'}]}


"Estás mirando una empresa llamada: Anthropic\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nHome \\ Anthropic\nContenido de la Web:\nSkip to main content\nSkip to footer\nResearch\nOverview\nAlignment\nEconomics\nEngineering\nFrontier Red Team\nInterpretability\nScience\nSocietal Impacts\nPolicy\nCommitments\nInitiatives\nClaude's Constitution\nClaude Corps\nPolicy on the AI Exponential\nTransparency\nResponsible Scaling Policy\nBeneficial Deployments\nTrust center\nSecurity and compliance\nLearn\nLearn\nClaude Academy\nTutorials\nUse cases\nDeveloper docs\nCompany\nAbout\nLeadership\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nAbout Claude\nOverview\nPricing\nContact sales\nModels\nMythos\nFable\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog

In [26]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [27]:
create_brochure("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página Acerca de nosotros', 'url': 'https://www.anthropic.com'}, {'type': 'Página de eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de noticias', 'url': 'https://www.anthropic.com/news'}]}


# 📜 Folleto de Anthropic: La Compañía que Hace que la IA Sea Genial (y Segura) 🌍🤖

**¡Bienvenidos al futuro, amigos!** Anthropic es donde la inteligencia artificial (IA) se encuentra con la seguridad, el buen humor y una pizca de locura. Únete a nosotros mientras navegamos por el vasto océano de la IA, ¡con la certeza de que no nos hundiremos!

---

## 🏢 Sobre Anthropic

En Anthropic, **creamos sistemas de IA confiables, interpretables y controlables**. Nos gusta pensar que somos los héroes de la película de IA, dedicados a asegurar que los beneficios de la IA se distribuyan y sus riesgos se minimicen. Y te prometemos que no llevamos capa... ¡ni trajes de lycra!

---

## 🎉 Cultura de Empresa:Una Fiesta de Inclusividad

1. **Actuamos por el bien global**: Solo queremos que nuestro trabajo (y el tuyo) deje huella, ¡y no, no hablamos de huellas dactilares!
   
2. **Calculamos luz y sombra**: La IA puede ser un superhéroe o un villano. Nosotros hacemos lo posible para que tu IA sea más "Superman" que "Lex Luthor".
   
3. **¡Sé bueno con los usuarios!**: Nuestros usuarios son nuestra familia (incluso si son un poco raros a veces).

4. **Fomentamos una carrera hacia el top en seguridad**: Queremos que todos los desarrolladores se esfuercen por crear sistemas de IA más seguros. ¡Es como la serie de juegos de "Gears of War", pero con algoritmos!

5. **Hacemos lo simple que funciona**: No intentamos reinventar la rueda cuando simplemente podemos darle un buen empujón.

6. **Seamos honestos y útiles**: En Anthropic, la confianza y la honestidad son la base de todo. Eso sí, los chistes malos deben ser ofrecidos sin arrepentimientos.

7. **Ponemos la misión primero**: Cuando se trata de cambiar el futuro de la IA, todos somos superhéroes (aunque no tengamos capa).

---

## 🛠️ Nuestros Productos: ¡Claude a la Vista!

Con Claude en el volante, el viaje va de maravilla. Desde **Claude Code** hasta **Claude for Microsoft 365**, tenemos todo lo que necesitas para navegar en este mar de tecnología. ¡Es como Disneylandia, pero en lugar de Mickey, tenemos inteligencia artificial!

---

## 🚀 Carreras en Anthropic: ¿Listo para unirte?

Si eres un investigador, ingeniero o simplemente alguien que ama resolver problemas del mundo real, ¡estás en el lugar correcto! Aquí en Anthropic, valoramos la diversidad de pensamientos y experiencias, porque sí, los sueños de IA pueden hacerse realidad cuando trabajamos juntos.

### Beneficios resaltantes:
- *22 semanas de licencia parental* (¡Porque las familias son lo primero!)
- *Stipendios de bienestar* ($500 al mes para mimarte un poco)
- ¡Snacks y comidas diarias! (Nadie trabaja con el estómago vacío, ¡cierto?)

---

## 💡 Únete a Nuestra Aventura

¿Listo para dar el siguiente paso y unirte a nuestro equipo de héroes de la IA? Consulta nuestras vacantes en [Anthropic Careers](https://anthropic.com/careers) y descubre cómo puedes ayudarnos a hacer del mundo un lugar más seguro y más divertido.

¡Nos encantaría tenerte a bordo!

---

**Anthropic: Donde la IA se encuentra con la seguridad, la diversión y un poquito de locura.** 🌐✨

In [19]:
create_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Página de Cursos', 'url': 'https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy'}, {'type': 'Página Rutas de Aprendizaje', 'url': 'https://cursos.frogamesformacion.com/pages/rutas-de-aprendizaje'}, {'type': 'Página de Proyectos de Evaluación', 'url': 'https://cursos.frogamesformacion.com/pages/proyectos-evaluacion'}, {'type': 'Página de Instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Página de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Página de Blog', 'url': 'https://cursos.frogamesformacion.com/pages/blog'}, {'type': 'Página Sobre Nosotros', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Página para Empresas', 'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'}, {'type': 'Página de Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Pá

# Folleto de Frogames Formación

## ¿Quiénes somos?
**Frogames Formación** es una plataforma de educación online especializada en la enseñanza de programación, desarrollo de videojuegos, matemáticas e inteligencia artificial. Fundada por **Juan Gabriel Gomila**, un instructor reconocido en Udemy con más de 600,000 estudiantes, nuestro enfoque es proporcionar formación de alta calidad de manera accesible.

## Nuestros cursos
Con más de **245 cursos** y **5954 horas de contenido** disponibles, ofrecemos **18 rutas de aprendizaje** que van de cero a experto en diversas áreas como:

- **Desarrollo de Videojuegos** (Unity, Unreal y Godot)  
- **Inteligencia Artificial** (redes neuronales, LLMs)  
- **Business Intelligence** (Power BI, Tableau)  
- **Análisis de Datos** (Python, SQL)  
- **Desarrollo Web** (HTML, CSS, JavaScript)  

Cada curso culmina con un **certificado verificable**, ideal para añadir a tu currículum o LinkedIn.

## Comunidad y Cultura
En Frogames, fomentamos una comunidad activa donde nuestros estudiantes pueden compartir experiencias, resolver dudas y colaborar. Valoramos:

- **Aprender mientras te diviertes**: Nuestros cursos utilizan técnicas interactivas para hacer el aprendizaje ameno.
- **Instructores Expertos**: Contamos con un equipo de profesionales que guían a nuestros alumnos durante su proceso de aprendizaje.
- **Actualizaciones Constantes**: Cada semana añadimos nuevos cursos y actualizaciones para mantenernos alineados con las últimas tendencias.

## Testimonios
> **"Me encanta aprender en Frogames porque sus cursos son especializados. ¡Siempre que ingreso, estoy con una sonrisa de oreja a oreja!"** - Edwyn Mendoza  

> **"Mi ilusión diaria es mi curso de Frogames. La calidad de su equipo humano es la mejor que he visto hasta ahora en cursos online."** - Toni Peña Palencia

## Oportunidades de carrera
Frogames no solo ofrece cursos, sino también **posibilidades de carrera** para sus alumnos. Trabajamos constantemente para conectar talentos con empresas que buscan profesionales capacitados en tecnología y programación.

## Convalidación de cursos
Si ya has tomado un curso en Udemy, puedes **convalidarlo con nosotros** y acceder a la formación actualizada junto con un certificado blockchain. 

## Planes de suscripción
Ofrecemos opciones flexibles para acceder a todo nuestro catálogo:
- **Rana de Bronce**: 39 €/mes  
- **Rana de Plata**: 199 €/6 meses  
- **Rana de Oro**: 349 €/año  

Cada suscripción da acceso a todos los cursos, rutas de aprendizaje y recursos adicionales.

## Conclusión
En Frogames Formación, estamos comprometidos con tu aprendizaje y desarrollo profesional. Únete a nuestra comunidad y empieza a adquirir habilidades que te abrirán puertas en el futuro laboral.

¿Listo para aprender? **Visítanos en [Frogames Formación](https://www.frogamesformacion.com)** y comienza tu viaje hoy mismo.

## Por último, una pequeña mejora

Con un pequeño ajuste, podemos cambiar esto para que los resultados se transmitan desde OpenAI,
con la animación de máquina de escribir habitual


In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina de liderazgo', 'url': 'https://www.anthropic.com/company/leadership'}, {'type': 'Carreras/empleos disponibles', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina de noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Pagina de Cursos', 'url': 'https://academy.claude.com'}]}



# Folleto de Anthropic

## Quiénes Somos
Anthropic es una corporación pública dedicada a la investigación y desarrollo de inteligencia artificial (IA) segura y responsable. Nos comprometemos a construir sistemas AI que sean **reliables**, **interpretables** y **dirigibles** con el objetivo de maximizar los beneficios de la IA y mitigar los riesgos asociados. Nuestro asistente AI, **Claude**, es un ejemplo de nuestro esfuerzo por crear tecnologías que beneficien a la humanidad a largo plazo.

## Nuestra Misión
Creemos que la IA tendrá un impacto profundo y duradero en el mundo. Nuestro propósito es asegurar que esta transición se realice de manera beneficiosa y segura, enfocándonos en:

- **Investigación de frontera en IA**: Exploramos diversas áreas de investigación, incluyendo interpretabilidad, política, y análisis de impacto social.
- **Colaboración interdisciplinaria**: Nuestro equipo está compuesto por investigadores, ingenieros, expertos en políticas y líderes empresariales, todos comprometidos con nuestra misión común.

## Cultura de la Empresa
### Valores que nos Guían
1. **Actuar por el bien global**: Tomamos decisiones que favorecen resultados positivos para la humanidad.
2. **Sostener luz y sombra**: Equilibramos el potencial riesgo y beneficio de la IA.
3. **Ser buenos con nuestros usuarios**: Fomentamos la generosidad y la amabilidad en todas nuestras interacciones.
4. **Iniciar una carrera hacia la seguridad**: Inspiramos la competencia para desarrollar sistemas de IA más seguros y confiables.
5. **Hacer lo simple que funciona**: Adoptamos un enfoque empírico para resolver problemas.
6. **Ser útiles, honestos e inofensivos**: Promovemos una comunicación abierta y de confianza dentro de la empresa.
7. **Poner la misión primero**: La misión es nuestra guía para decisiones y acciones.

## Productos y Servicios
Anthropic ofrece una variedad de modelos de IA a través de **Claude**, incluyendo:

- **Claude Code**
- **Claude Science**
- **Claude Security** 
- **Claude para Microsoft 365**
- Soluciones para diversas industrias como educación, atención médica, finanzas y más.

### Claude Academy
Proporcionamos **cursos y tutoriales** a través de Claude Academy, donde users pueden aprender a utilizar nuestros modelos y desarrollar su propio potencial en el sector de IA.

## Oportunidades de Carrera
En Anthropic, siempre estamos buscando personas apasionadas para unirse a nuestro equipo. Nuestra cultura es inclusiva y valoramos las diversas trayectorias profesionales. Si eres una persona que busca enfrentar problemas desafiantes con alta relevancia social, ¡queremos conocerte!

### Posiciones Abiertas
- Investigadores en IA
- Ingenieros de software
- Especialistas en políticas
- Técnicos en seguridad

Visita nuestra página de **[Carreras](#)** para más información sobre las posiciones disponibles.

## Clientes y Socios
Trabajamos con empresas, organizaciones sin fines de lucro y entidades gubernamentales para desplegar tecnologías de IA que generen un impacto positivo en sus comunidades. Algunas de nuestras soluciones están diseñadas específicamente para sectores como la salud, finanzas y educación.

## Contáctanos
Para más información sobre nuestras iniciativas, productos o carreras, visita [Anthropic](https://www.anthropic.com) o contáctanos en nuestras redes sociales.

---

**Anthropic PBC**  
© 2026 Anthropic. Todos los derechos reservados.



In [29]:
stream_brochure("HuggingFace", "https://huggingface.co")

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://huggingface.co/huggingface'}, {'type': 'Página de Carreras', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Página de Cursos', 'url': 'https://huggingface.co/learn'}, {'type': 'Página de Modelos', 'url': 'https://huggingface.co/models'}, {'type': 'Página de Datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'Página de Espacios', 'url': 'https://huggingface.co/spaces'}, {'type': 'Página de Blog', 'url': 'https://huggingface.co/blog'}, {'type': 'Página de Precios', 'url': 'https://huggingface.co/pricing'}, {'type': 'Página de Documentación', 'url': 'https://huggingface.co/docs'}, {'type': 'Página de Soporte', 'url': 'https://huggingface.co/support'}]}



# 🐻 Hugging Face: La Comunidad AI que Edifica el Futuro

¡Hola, futuros colaboradores, inversores y curiosos! Si alguna vez soñaron con un lugar donde la inteligencia artificial se une con la magia del aprendizaje automático, ¡bienvenidos a Hugging Face! Aquí, abrazamos la tecnología como si fuera un enorme oso de peluche, no solo para sentirnos bien, sino para construir el futuro de la IA. 

## 🚀 ¿Quiénes Somos?

### La Misión
Estamos en una cruzada para **democratizar** el MX (Machine Learning), un commit a la vez. Somos una comunidad vibrante de más de 50,000 organizaciones (sí, escucharon bien) que incluyen gigantes como Amazon y Google. Juntos estamos en la vanguardia de la innovación.

### ¿Qué Hacemos?
- **Modelos**: ¡Más de 2 millones de modelos listos para ser usados! Desde generación de texto hasta imágenes, cubrimos todo el espectro.
- **Datasets**: Con 500,000+ datasets, ¡nunca tendrás que lidiar con la "nada que aprender"! 
- **Espacios**: Perfectos para alojar y colaborar en aplicaciones de IA sin romper el banco.

## 🎓 Cursos y Packs para Futuras Estrellas de la IA

¿Te gustaría ser el próximo Einstein del aprendizaje automático? ¡En Hugging Face tenemos un calendario lleno de cursos! Aquí solo algunos:

- **Curso de LLM**: Aprende sobre los modelos de lenguaje (Large Language Models) utilizando nuestras bibliotecas.
- **Curso de Robótica**: Construye robots, ¡y quién sabe, tal vez el próximo R2-D2!
- **Curso Mágico "Smol"**: ¡La aventura más pequeña en el mundo de los modelos de IA post-entrenamiento!

🎉 **¿Todo suena increíble?** Únete a nuestra comunidad en **Discord** para obtener consejos de otros genios de IA.

## 🌈 La Cultura de Hugging Face

Imagínate un entorno de trabajo donde los memes sobre IA son tan comunes como el café a media mañana. Aquí, la creatividad choca con el conocimiento, ¡y todos se apoyan en el aprendizaje colectivo! Cada semana, celebramos los éxitos y, claro, reímos juntos de nuestras fallas en un ambiente relajado.

## 🏆 Únete a Nosotros

Si todas estas cosas resuenan contigo y quieres ser parte de una revolución de IA, ¡mira nuestras ofertas de empleo! Estamos buscando mentes curiosas, desde magos de la programación hasta soñadores que pueden mirar hacia el futuro de la IA y decir: "¡Yo puedo mejorar eso!"

## 🌟 En Resumen 

Hugging Face no es solo una plataforma; es el hogar de aquellos que desean hacer de la inteligencia artificial una fuerza para el bien. Con acceso a modelos, datasets, y comunidades increíbles, no solo construirás tu carrera, sino que construirás el futuro.

¡Haz el primer paso, únete a Hugging Face y **abracemos** juntos la IA!

---

*¿Listo para empezar? Descubre más en [Hugging Face](https://huggingface.co) y únete a la comunidad que está haciendo historia en la IA.*


In [23]:
stream_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Pagina Principal', 'url': 'https://cursos.frogamesformacion.com/'}, {'type': 'Pagina de Cursos', 'url': 'https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy'}, {'type': 'Pagina de Rutas', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Pagina de Instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Pagina de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Rutas de Aprendizaje', 'url': 'https://cursos.frogamesformacion.com/pages/rutas-de-aprendizaje'}, {'type': 'Proyectos de Evaluación', 'url': 'https://cursos.frogamesformacion.com/pages/proyectos-evaluacion'}, {'type': 'Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Nuestros Clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Frogames para Empresas', 'url': 'https://cursos.frogamesformaci

# Frogames Formación

## Quiénes somos
Frogames Formación es una plataforma de educación online dedicada a ofrecer cursos de programación, desarrollo de videojuegos, inteligencia artificial y más, todos impartidos en español. Con más de **708,820 estudiantes** y **245 cursos** disponibles, nuestra misión es proporcionar una educación accesible y de alta calidad.

## Formación de calidad

### Cursos y Rutas de Aprendizaje
- **Variadas áreas**: Desde desarrollo de videojuegos (Unity, Unreal, Godot), pasando por programación en Android e iOS, hasta temas como inteligencia artificial y análisis de datos.
- **18 rutas de aprendizaje**: Estructuradas para llevarte de principiante a experto, cada ruta está diseñada para maximizar tu aprendizaje y dominio de la materia.
- **Certificaciones Verificables**: Al finalizar un curso, recibirás un certificado digital que podrás compartir en tu CV o en LinkedIn.

### Nuestras estadísticas
- **5954 horas de contenido**
- **4.99 ★** (en 4802 reseñas)

## Cultura de la empresa
Frogames fomenta un ambiente inclusivo y colaborativo. Nuestro equipo de instructores expertos son apasionados por la enseñanza, dispuestos a acompañar a los estudiantes a lo largo de su aprendizaje. Valoramos la comunidad, y nuestros estudiantes destacan el apoyo constante y la calidad de los cursos.

### Historias de Éxito
Muchos de nuestros estudiantes han encontrado en Frogames el lugar ideal para desarrollarse tanto personalmente como profesionalmente. Desde jóvenes que inician su recorrido en la programación hasta adultos que buscan un cambio en su vida laboral, todos encuentran un espacio en nuestra plataforma.

## Oportunidades de Carrera
Frogames no solo se enfoca en la enseñanza. También valoramos el crecimiento profesional de nuestro equipo. Estamos en constante búsqueda de nuevos talentos para unirse a nuestro equipo. Si te apasiona la educación y la tecnología, ¡considera unirte a nosotros!

## Paquetes y Suscripciones
- **Rana de Bronce**: 39 €/mes
- **Rana de Plata**: 199 €/6 meses
- **Rana de Oro**: 349 €/año
- **Rana Black**: 7999 € (pago único, acceso ilimitado)

Con cualquiera de nuestras suscripciones, podrás acceder a toda la biblioteca de cursos y recursos disponibles, incluyendo actualizaciones semanales de contenido.

## Conclusión
Únete a la comunidad de Frogames Formación y comienza tu viaje hacia la adquisición de nuevas habilidades. Con cursos para todos los niveles y un fuerte enfoque en el aprendizaje práctico, ¡hay un lugar para ti aquí!

Para más información, visita nuestro sitio web y comienza a aprender hoy mismo.

## Aplicaciones empresariales

En este ejercicio, ampliamos el código del día 1 para realizar múltiples llamadas a LLM y generar un documento.

En términos de técnicas, este es quizás el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLM. Esto se abordará más en la semana 2 y luego volveremos a Agentic AI de manera importante en la semana 8, cuando construyamos una solución Agent completamente autónoma.

En términos de aplicaciones empresariales, generar contenido de esta manera es uno de los casos de uso más comunes. Al igual que con el resumen, esto se puede aplicar a cualquier vertical empresarial. Escriba contenido de marketing, genere un tutorial de producto a partir de una especificación, cree contenido de correo electrónico personalizado y mucho más. Explore cómo puede aplicar la generación de contenido a su negocio e intente crear un prototipo de prueba de concepto.